In [3]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv() 

PROVIDER = "groq"  

LLM_CONFIGS = {
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",  
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": os.getenv("OPENAI_API_KEY"),
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}

def get_llm(temperature: float = 0.5, seed: int | None = None) -> ChatOpenAI:
    config = LLM_CONFIGS[PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, seed=seed, max_retries=5)

llm = get_llm(temperature=0.7)
print(f"{PROVIDER} / {llm.model_name}")

groq / openai/gpt-oss-120b


In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


cook_prompt = PromptTemplate.from_template(
     """당신은 친절한 AI 요리사입니다.
사용자가 가진 재료로 만들 수 있는 요리를 1가지 추천하고, 간단한 레시피를 알려주세요.
재료 목록에 없는 주재료는 새로 추가하지 마세요. (기름, 소금 같은 기본 조미료는 가능)

재료: {ingredients}

아래 형식으로 한국어로 답변하세요.

{ingredients}로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (사용하는 재료)
조리법:
1. ...
2. ...
3. ..."""
)

cook_chain = cook_prompt | llm | StrOutputParser()

print(cook_chain.invoke({"ingredients": "계란, 밥, 김치"}))

계란, 밥, 김치로 만들 수 있는 요리를 추천드립니다!

**추천 요리:** 김치볶음밥 + 계란프라이  

**재료:**  
- 밥 1공기  
- 김치 1/2컵 (잘게 썬 것)  
- 계란 1개  
- 식용유 2큰술  
- 소금 약간  
- 참기름 1작은술 (선택)  
- 깨소금 약간 (선택)  

**조리법:**  
1. **재료 준비**  
   - 김치는 물기를 살짝 짜고 작게 다진다.  
   - 밥은 고슬고슬하게 풀어 놓는다.  

2. **김치볶음밥 만들기**  
   1) 팬에 식용유 1큰술을 두르고 중불에 달군다.  
   2) 다진 김치를 넣고 2~3분 정도 볶아 김치가 살짝 투명해지면 소금으로 간을 맞춘다.  
   3) 밥을 넣고 김치와 골고루 섞이도록 뒤집어가며 3~4분 정도 볶는다.  
   4) 마지막에 참기름 1작은술과 깨소금을 넣어 한 번 더 섞어 주면 완성.  

3. **계란프라이 만들기**  
   1) 같은 팬에 남은 식용유 1큰술을 추가하고 중약불로 예열한다.  
   2) 계란을 깨뜨려 넣고 흰자가 반쯤 익을 때까지 살짝 뒤집지 않는다.  
   3) 노른자가 살짝 반숙이 되면 불을 끄고 팬을 살짝 흔들어 기름을 고르게 만든다.  

4. **플레이팅**  
   - 접시에 김치볶음밥을 옮겨 담고, 그 위에 계란프라이를 올린다.  
   - 원한다면 추가로 깨소금을 뿌리거나 고추가루를 살짝 뿌려도 좋다.  

**팁**  
- 밥이 차갑다면 전자레인지에 살짝 데워두면 볶을 때 뭉치지 않는다.  
- 김치의 매운 정도에 따라 소금 양을 조절한다.  

맛있게 즐기세요! 🍚🥚🌶️


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

parser = StrOutputParser()

# 1단계
spot_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 전문가입니다. 입력된 도시나 국가의 대표 관광 명소를 딱 1가지만 추천합니다. "
               "설명 없이 명소 이름만 한 줄로 답하세요."),
    ("user", "여행지: {destination}"),
])

# 2단계
detail_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 아래 형식을 지켜 한국어로 답하세요.\n\n"
               "명소: (명소 이름)\n역사:\n(역사 설명)\n특징:\n(특징 설명)\n방문 팁:\n(방문 팁)"),
    ("user", "{spot}에 대한 상세 정보를 알려주세요."),
])

spot_chain = spot_prompt | llm | parser
detail_chain = detail_prompt | llm | parser


travel_chain = (
    RunnablePassthrough.assign(spot=spot_chain)
    | RunnablePassthrough.assign(detail=detail_chain)
)


destination = "로마"
result = travel_chain.invoke({"destination": destination})
print(f"입력: {destination}")
print(f"1단계 결과(추천 명소): {result['spot']}\n")
print("2단계 결과(상세 정보):")
print(result["detail"])

입력: 로마
1단계 결과(추천 명소): 콜로세움

2단계 결과(상세 정보):
명소: 콜로세움 (로마 콜로세움, Amphitheatrum Flavium)

역사:  
콜로세움은 서기 72년 베스파시아누스 황제가 착공하고, 그의 아들 티투스 황제가 80년에 완공한 고대 로마의 원형극장이다. 플라비우스 왕조(Flavian dynasty)의 이름을 따서 ‘플라비우스 원형극장’이라고도 불린다. 5만~8만 명을 수용할 수 있었으며, 검투사 경기, 야생동물 사냥, 해전 모의전투(가상 해전) 등 다양한 공개 오락을 위해 사용되었다. 서기 5세기 이후 서서히 사용이 중단되고, 중세에는 성벽이나 집단 거주지로 개조되었으며, 18·19세기에는 문화재 보호 운동의 상징이 되었다. 현재는 유네스코 세계문화유산에 등재된 로마의 대표적인 고대 건축물이다.

특징:  
- **구조와 설계**: 80m 높이, 외경 약 189m, 내부 타원형 경기장은 87m × 55m 크기이며, 4층(지하층 포함)으로 구성된 복합 구조다. 아치와 콘크리트, 석재를 활용한 혁신적인 건축 기술이 적용되었다.  
- **입구와 좌석**: 총 80개의 입구가 있어 관객 흐름을 효율적으로 관리했으며, 사회적 신분에 따라 좌석 구역이 엄격히 구분되었다(귀족, 시민, 노예·여성 등).  
- **하부 구조**: 지하에 ‘히포게움’이라 불리는 복잡한 통로와 방이 있었으며, 여기서 동물과 검투사, 무대 장치를 준비했다.  
- **복원과 보수**: 1990년대 이후 지속적인 복원 작업이 진행돼 현재는 원형극장의 외관이 비교적 잘 보존돼 있다. 야간 조명과 레이저 쇼가 추가돼 관광객에게 새로운 경험을 제공한다.

방문 팁:  
1. **입장권 사전 예매**: 온라인으로 사전 예약하면 긴 대기 시간을 피할 수 있다. ‘콜로세움 + 포럼 + 팔라티노 언덕’ 복합 티켓을 구매하면 비용을 절감할 수 있다.  
2. **조조·조금 늦은 오후**: 개장 시간은 보통 오전 8시~오후 7시(시즌에 따라 변동). 조조에 입장하면 인파가 적고

In [7]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])


examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. "
                "이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, "
                "세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. "
                "전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    {  # 추가 예시
        "news": "한국은행이 기준금리를 동결했다. 물가 상승세가 둔화됐지만 가계부채와 환율 불안이 "
                "계속되고 있어 당분간 신중한 통화정책을 이어가겠다는 입장이다.",
        "keywords": "한국은행, 기준금리, 가계부채",
    },
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 뉴스에서 핵심 키워드 3개를 추출하세요. "
               "반드시 '키워드: A, B, C' 형식으로만 답하세요."),
    few_shot_prompt,
    ("human", "{input}"),
])

keyword_chain = final_prompt | get_llm(temperature=0, seed=42) | StrOutputParser()

test_news = (
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 "
    "제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 "
    "사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 "
    "제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."
)
print(keyword_chain.invoke({"input": test_news}))

test_set = {
    "스포츠": "프로야구 정규시즌 최종전에서 한화 이글스가 극적인 역전승을 거두며 포스트시즌 진출을 확정지었다. "
              "마지막 이닝에 터진 끝내기 안타가 승부를 갈랐다.",
    "환경": "정부가 2035년까지 석탄발전을 단계적으로 줄이고 재생에너지 비중을 대폭 늘리는 탄소중립 로드맵을 발표했다. "
            "태양광과 풍력 설비 확대가 핵심이다.",
    "교육": "교육부가 내년부터 초등학교 전 학년에 디지털 교과서를 도입한다고 밝혔다. "
            "교사 연수와 학습 격차 해소 방안도 함께 추진된다.",
}
results = keyword_chain.batch([{"input": text} for text in test_set.values()])
for field, r in zip(test_set, results):
    print(f"[{field}] {r}")

키워드: 제미나이, 구글 AI 스튜디오, 멀티모달
[스포츠] 키워드: 한화 이글스, 역전승, 포스트시즌
[환경] 키워드: 탄소중립, 석탄발전, 재생에너지
[교육] 키워드: 교육부, 디지털 교과서, 학습 격차
